# 12 — KalmanNet Held-Out Test Evaluation & Ablation

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Roadmap Section 19:** Testing notebook must load the saved checkpoint and evaluate on held-out data.
> **Held-Out Test Session:** Unseen Driver A session `S1`.
> **Benchmark Comparison:** Classical ESKF vs Fixed Gain ($K = 0.80$) vs KalmanNet (Learned Dynamic Gain).

### Objectives:
1. Load trained KalmanNet from `checkpoints/kalmannet/kalmannet_best.pt`.
2. Execute continuous filtering across full held-out test route `S1`.
3. Compare trajectory tracking against Ground Truth (RTK/GPS reference).
4. Analyze Kalman Gain adaptation dynamics across straight vs cornering maneuvers.

## 1. Environment & Checkpoint Loading

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.kalmannet import KalmanNetNN
from src.datasets.kalmannet_dataset import KalmanNetDataset
from src.preprocessing.data_loader import IOVNBDLoader
from src.calibration.alignment import PhoneVehicleAlignment

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_path = PROJECT_ROOT / 'checkpoints' / 'kalmannet' / 'kalmannet_best.pt'
plots_dir = PROJECT_ROOT / 'plots' / 'kalmannet'
results_dir = PROJECT_ROOT / 'results'
plots_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print(f'Loading KalmanNet Checkpoint: {ckpt_path}')
if not ckpt_path.exists():
    raise FileNotFoundError(f'Checkpoint not found at {ckpt_path}. Run 11_kalmannet_training.ipynb first.')

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = ckpt.get('config', {})

model = KalmanNetNN(
    state_dim=cfg.get('state_dim', 4),
    meas_dim=cfg.get('meas_dim', 2),
    hidden_dim=cfg.get('hidden_dim', 64),
    num_layers=cfg.get('num_layers', 2)
).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'KalmanNet successfully loaded from Epoch {ckpt.get("epoch")} (Best Val Loss: {ckpt.get("best_val_loss"):.4f})')

## 2. Load Continuous Held-Out Test Route (Session S1 — Driver A)

In [ ]:
loader = IOVNBDLoader()
sess_name = 'S1'
print(f'Loading held-out test session: {sess_name}')
sess = loader.load_session(sess_name, preprocess_imu=True)

acc = sess['accel_filtered']
gyr = sess['gyro_filtered']
enu_gt = sess['enu_coords'][:, :2]  # (N, 2)
n_samples = len(enu_gt)
dt = 0.1

veh_spd = sess['vehicle']['speed_mps']
if veh_spd is None:
    veh_spd = sess['gps']['speed_mps']
if veh_spd is None:
    veh_spd = np.zeros(n_samples, dtype=np.float32)

# Phone-to-vehicle alignment
aligner = PhoneVehicleAlignment()
R_p_to_v = aligner.calibrate(sess['accel_raw'], zupt_mask=sess['zupt_mask'], velocity_ref=veh_spd)
acc_v = (R_p_to_v @ acc.T).T
gyr_v = (R_p_to_v @ gyr.T).T

# Compute GT velocity
vel_gt = np.zeros_like(enu_gt, dtype=np.float32)
vel_gt[1:-1] = (enu_gt[2:] - enu_gt[:-2]) / (2.0 * dt)
vel_gt[0] = (enu_gt[1] - enu_gt[0]) / dt
vel_gt[-1] = (enu_gt[-1] - enu_gt[-2]) / dt

# Heading from displacement
diff_enu = np.diff(enu_gt, axis=0, prepend=enu_gt[0:1])
headings = np.arctan2(diff_enu[:, 1], diff_enu[:, 0])

# Navigation-frame acceleration
c_h, s_h = np.cos(headings), np.sin(headings)
a_east = acc_v[:, 0] * c_h - acc_v[:, 1] * s_h
a_north = acc_v[:, 0] * s_h + acc_v[:, 1] * c_h
a_nav_all = np.stack([a_east, a_north], axis=1).astype(np.float32)

# Odometry Velocity Measurements z_meas [v_east, v_north]
io_ckpt = PROJECT_ROOT / 'checkpoints' / 'inertial_odometry' / 'inertial_odometry_best.pt'
z_meas_all = np.zeros((n_samples, 2), dtype=np.float32)
if io_ckpt.exists():
    try:
        from src.models.inertial_odometry import NeuralInertialOdometry
        io_dict = torch.load(io_ckpt, map_location=device, weights_only=False)
        io_model = NeuralInertialOdometry(
            input_dim=6,
            tcn_channels=io_dict.get('config', {}).get('tcn_channels', [64, 128, 256]),
            kernel_size=io_dict.get('config', {}).get('tcn_kernel_size', 3)
        ).to(device)
        io_model.load_state_dict(io_dict['model_state_dict'])
        io_model.eval()
        print('Loaded Phase 4 Neural Inertial Odometry model for test session.')
        imu_6d = np.hstack([acc_v, gyr_v]).astype(np.float32)
        pred_speeds = np.copy(veh_spd)
        with torch.no_grad():
            for ws in range(0, n_samples - 100 + 1, 10):
                we = ws + 100
                w_t = torch.from_numpy(imu_6d[ws:we]).unsqueeze(0).to(device)
                _, p_v, _ = io_model(w_t)
                pred_speeds[we - 1] = p_v[0, 0].item()
        z_meas_all[:, 0] = pred_speeds * c_h
        z_meas_all[:, 1] = pred_speeds * s_h
    except Exception as ex:
        print(f'Note: IO Model inference exception ({ex}), using reference velocity + noise.')
        noise = np.random.RandomState(101).normal(0, 0.4, size=n_samples)
        z_meas_all[:, 0] = np.maximum(0, veh_spd + noise) * c_h
        z_meas_all[:, 1] = np.maximum(0, veh_spd + noise) * s_h
else:
    noise = np.random.RandomState(101).normal(0, 0.4, size=n_samples)
    z_meas_all[:, 0] = np.maximum(0, veh_spd + noise) * c_h
    z_meas_all[:, 1] = np.maximum(0, veh_spd + noise) * s_h

total_dist = float(np.sum(np.linalg.norm(np.diff(enu_gt, axis=0), axis=1)))
print(f'Held-Out Session S1 Duration: {n_samples * dt:.1f}s ({n_samples} points) | Total Distance: {total_dist:.1f}m')

## 3. Execute Trajectory Estimation Benchmarks
We compare 4 filtering paradigms on the same continuous trajectory:
1. **Pure IMU Dead Reckoning**: Direct integration without velocity updates.
2. **Classical ESKF Baseline**: Fixed observation covariance without adaptive gain.
3. **Fixed Kalman Gain**: Hardcoded gain matrix $K = 0.80$ (demonstrating why fixed gain is inadequate).
4. **KalmanNet (Proposed)**: Recurrent neural filter learning dynamic, state-dependent Kalman gain.

In [ ]:
F_m = torch.tensor([
    [1.0, 0.0, dt,  0.0],
    [0.0, 1.0, 0.0, dt ],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0]
], dtype=torch.float32, device=device)

B_m = torch.tensor([
    [0.5 * dt**2, 0.0],
    [0.0, 0.5 * dt**2],
    [dt, 0.0],
    [0.0, dt]
], dtype=torch.float32, device=device)

H_m = torch.tensor([
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0]
], dtype=torch.float32, device=device)

# 1. Pure IMU Integration
pos_pure = np.zeros((n_samples, 2), dtype=np.float32)
vel_pure = np.zeros((n_samples, 2), dtype=np.float32)
pos_pure[0] = enu_gt[0]
vel_pure[0] = vel_gt[0]
for k in range(1, n_samples):
    pos_pure[k] = pos_pure[k-1] + vel_pure[k-1] * dt + 0.5 * a_nav_all[k] * (dt**2)
    vel_pure[k] = vel_pure[k-1] + a_nav_all[k] * dt

# 2. Fixed Gain Filter (K = 0.80 on velocity, K = 0.8 * dt on position)
pos_fixed = np.zeros((n_samples, 2), dtype=np.float32)
vel_fixed = np.zeros((n_samples, 2), dtype=np.float32)
pos_fixed[0] = enu_gt[0]
vel_fixed[0] = vel_gt[0]
for k in range(1, n_samples):
    p_prior = pos_fixed[k-1] + vel_fixed[k-1] * dt + 0.5 * a_nav_all[k] * (dt**2)
    v_prior = vel_fixed[k-1] + a_nav_all[k] * dt
    innov = z_meas_all[k] - v_prior
    pos_fixed[k] = p_prior + 0.08 * innov
    vel_fixed[k] = v_prior + 0.80 * innov

# 3. KalmanNet Rollout
pos_knet = np.zeros((n_samples, 2), dtype=np.float32)
vel_knet = np.zeros((n_samples, 2), dtype=np.float32)
k_gains_recorded = np.zeros((n_samples, 4, 2), dtype=np.float32)

pos_knet[0] = enu_gt[0]
vel_knet[0] = vel_gt[0]

x_curr = torch.tensor([enu_gt[0, 0], enu_gt[0, 1], vel_gt[0, 0], vel_gt[0, 1]], dtype=torch.float32, device=device).unsqueeze(0)
z_prev = torch.tensor(z_meas_all[0], dtype=torch.float32, device=device).unsqueeze(0)
h_prev = None

with torch.no_grad():
    for k in range(1, n_samples):
        a_t = torch.tensor(a_nav_all[k], dtype=torch.float32, device=device).unsqueeze(0)
        z_t = torch.tensor(z_meas_all[k], dtype=torch.float32, device=device).unsqueeze(0)

        x_prior = torch.matmul(x_curr, F_m.T) + torch.matmul(a_t, B_m.T)
        x_post, K_gain, h_prev = model.step(x_prior, z_t, H_m, x_curr, z_prev, h_prev)
        pos_knet[k] = x_post[0, 0:2].cpu().numpy()
        vel_knet[k] = x_post[0, 2:4].cpu().numpy()
        k_gains_recorded[k] = K_gain[0].cpu().numpy()
        x_curr = x_post
        z_prev = z_t

print('Trajectory rollouts completed for all benchmarks.')

## 4. Quantitative Benchmark Performance Comparison

In [ ]:
# Compute End-Point Error and Trajectory Drift
err_pure = np.linalg.norm(pos_pure - enu_gt, axis=1)
err_fixed = np.linalg.norm(pos_fixed - enu_gt, axis=1)
err_knet = np.linalg.norm(pos_knet - enu_gt, axis=1)

rmse_pure_pos = float(np.sqrt(np.mean(err_pure**2)))
rmse_fixed_pos = float(np.sqrt(np.mean(err_fixed**2)))
rmse_knet_pos = float(np.sqrt(np.mean(err_knet**2)))

final_drift_pure = float(err_pure[-1])
final_drift_fixed = float(err_fixed[-1])
final_drift_knet = float(err_knet[-1])

drift_pct_pure = (final_drift_pure / total_dist) * 100.0
drift_pct_fixed = (final_drift_fixed / total_dist) * 100.0
drift_pct_knet = (final_drift_knet / total_dist) * 100.0

# Velocity RMSE
v_err_fixed = np.linalg.norm(vel_fixed - vel_gt, axis=1)
v_err_knet = np.linalg.norm(vel_knet - vel_gt, axis=1)
rmse_fixed_vel = float(np.sqrt(np.mean(v_err_fixed**2)))
rmse_knet_vel = float(np.sqrt(np.mean(v_err_knet**2)))

print('=' * 75)
print('  HELD-OUT TEST SESSION S1 BENCHMARK SUMMARY')
print('=' * 75)
print(f'Method                    | Pos RMSE (m) | Final Drift (m) | Drift % (Target <10%)')
print('-' * 75)
print(f'Pure IMU Dead Reckoning   | {rmse_pure_pos:12.2f} | {final_drift_pure:15.2f} | {drift_pct_pure:18.2f}%')
print(f'Fixed Gain (K = 0.80)     | {rmse_fixed_pos:12.2f} | {final_drift_fixed:15.2f} | {drift_pct_fixed:18.2f}%')
print(f'KalmanNet (Proposed)      | {rmse_knet_pos:12.2f} | {final_drift_knet:15.2f} | {drift_pct_knet:18.2f}%')
print('=' * 75)
print(f'KalmanNet Velocity RMSE   : {rmse_knet_vel:.3f} m/s (vs Fixed Gain: {rmse_fixed_vel:.3f} m/s)')

## 5. Diagnostic Visualizations

In [ ]:
# 1. 2D Trajectory Comparison Plot
plt.figure(figsize=(10, 8))
plt.plot(enu_gt[:, 0], enu_gt[:, 1], 'k-', lw=2.5, label='Ground Truth (RTK/GPS)')
plt.plot(pos_knet[:, 0], pos_knet[:, 1], 'g-', lw=2.0, label=f'KalmanNet (Drift: {drift_pct_knet:.1f}%)')
plt.plot(pos_fixed[:, 0], pos_fixed[:, 1], 'm--', lw=1.8, label=f'Fixed Gain K=0.80 (Drift: {drift_pct_fixed:.1f}%)')
plt.plot(enu_gt[0, 0], enu_gt[0, 1], 'go', markersize=10, label='Start')
plt.plot(enu_gt[-1, 0], enu_gt[-1, 1], 'rx', markersize=10, mew=2.5, label='End')
plt.xlabel('East Position (meters)')
plt.ylabel('North Position (meters)')
plt.title('Held-Out Test Session S1: 2D Dead Reckoning Trajectory Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
traj_path = plots_dir / 'kalmannet_trajectory_comparison_S1.png'
plt.savefig(traj_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved trajectory comparison: {traj_path}')

# 2. Kalman Gain Adaptation Dynamics
t_sec = np.arange(n_samples) * dt
plt.figure(figsize=(12, 7))

plt.subplot(3, 1, 1)
plt.plot(t_sec, veh_spd, 'b-', lw=1.5, label='Vehicle Speed (m/s)')
plt.ylabel('Speed (m/s)')
plt.title('KalmanNet Gain Adaptation vs Vehicle Dynamics (Session S1)')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(3, 1, 2)
yaw_rate = np.abs(np.gradient(headings, dt))
plt.plot(t_sec, yaw_rate, 'orange', lw=1.5, label='Yaw Rate |omega_z| (rad/s)')
plt.ylabel('Yaw Rate (rad/s)')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(3, 1, 3)
plt.plot(t_sec, k_gains_recorded[:, 2, 0], 'g-', lw=1.5, label='K[ve, ve] (Velocity Gain)')
plt.plot(t_sec, k_gains_recorded[:, 0, 0], 'purple', lw=1.2, ls='--', label='K[pe, ve] (Position Injection Gain)')
plt.axhline(0.80, color='r', ls=':', label='Prohibited Fixed Gain (K=0.80)')
plt.ylabel('Kalman Gain')
plt.xlabel('Time (seconds)')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
gain_path = plots_dir / 'kalmannet_gain_adaptation_S1.png'
plt.savefig(gain_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved gain adaptation diagnostics: {gain_path}')

# Export Results JSON
results_data = {
    'session': 'S1',
    'driver': 'Driver A',
    'total_distance_m': total_dist,
    'duration_s': float(n_samples * dt),
    'rmse_pure_pos_m': rmse_pure_pos,
    'rmse_fixed_pos_m': rmse_fixed_pos,
    'rmse_knet_pos_m': rmse_knet_pos,
    'final_drift_pure_m': final_drift_pure,
    'final_drift_fixed_m': final_drift_fixed,
    'final_drift_knet_m': final_drift_knet,
    'drift_pct_pure': drift_pct_pure,
    'drift_pct_fixed': drift_pct_fixed,
    'drift_pct_knet': drift_pct_knet,
    'rmse_knet_vel_mps': rmse_knet_vel,
    'kalman_gain_stats': {
        'mean_k_ve': float(np.mean(k_gains_recorded[:, 2, 0])),
        'std_k_ve': float(np.std(k_gains_recorded[:, 2, 0])),
        'min_k_ve': float(np.min(k_gains_recorded[:, 2, 0])),
        'max_k_ve': float(np.max(k_gains_recorded[:, 2, 0]))
    }
}

res_file = results_dir / 'kalmannet_results.json'
with open(res_file, 'w') as f:
    json.dump(results_data, f, indent=2)
print(f'Test results exported to: {res_file}')